# WorldCupPool (Exploratory Companion)

This notebook is an exploratory companion to the script-first workflow in this repository.
For reproducible runs, use `BetfairBayesianScoreline.py` and `Method_Comparison.py` as the canonical entry points.

---

# The Philosophy Behind the Simulation: Game Theory in Mega-Pools


In [ ]:
import numpy as np

def simulate_full_tournament(m_simulations=10000, n_competitors_list=[10, 50, 100, 500, 1000, 10000]):
    n_matches = 64
    
    # 1. Generate realistic match parameters (Group + Knockout stage mix)
    np.random.seed(42) # Seeded for reproducible variance
    lambdas_H = 1.2 
    lambdas_A = 0.6 
    
    # 2. My predictions: The Poisson Mode for every match
    my_H_pred = np.floor(lambdas_H).astype(int)
    my_A_pred = np.floor(lambdas_A).astype(int)
    my_res_pred = np.sign(my_H_pred - my_A_pred)  # 1 = Home Win, 0 = Draw, -1 = Away Win

    # 3. Public Pool predictions (Biased toward favorites & specific scores)
    human_picks = [
        (2, 0, 0.25), (2, 1, 0.25), (1, 0, 0.15), 
        (3, 0, 0.10), (3, 1, 0.10), (1, 1, 0.10), (0, 0, 0.05)
    ]
    pick_probs = [p[2] for p in human_picks]
    human_f = np.array([p[0] for p in human_picks])
    human_a = np.array([p[1] for p in human_picks])
    human_res = np.sign(human_f - human_a)

    print(f"{'Opponents (N)':<15} | {'Outright Win %':<15} | {'Mean Rank Percentile':<20}")
    print("-" * 55)

    # 4. Scale and Evaluate over N competitors
    for n in n_competitors_list:
        my_total_points = np.zeros(m_simulations, dtype=int)
        opp_total_points = np.zeros((n, m_simulations), dtype=int)
        
        # Accumulate points match by match to preserve memory
        for match_idx in range(n_matches):
            # True outcomes for this specific match across M simulations
            true_h = np.random.poisson(lambdas_H, m_simulations)
            true_a = np.random.poisson(lambdas_A, m_simulations)
            true_res = np.sign(true_h - true_a)
            
            # My points for this match
            my_pts = ((true_res == my_res_pred) * 3) + \
                     ((true_h == my_H_pred) & (true_a == my_A_pred)) * 2
            my_total_points += my_pts
            
            # Precompute points for all 7 human pick types for this match
            pick_pts = ((true_res == human_res[:, None]) * 3) + \
                       ((true_h == human_f[:, None]) & (true_a == human_a[:, None])) * 2
            
            # Assign biased picks to N opponents
            opp_choices = np.random.choice(len(human_picks), size=(n, m_simulations), p=pick_probs)
            opp_total_points += pick_pts[opp_choices, np.arange(m_simulations)]

        # 5. Determine Rank and Win Rate
        opp_max_points = np.max(opp_total_points, axis=0)
        win_rate = np.mean(my_total_points > opp_max_points) * 100
        
        # Rank calculation: How many opponents strictly outscored me?
        opp_better = np.sum(opp_total_points > my_total_points, axis=0)
        my_rank = opp_better + 1 
        
        # Percentile rank (100% = First Place, 0% = Last Place)
        percentile_rank = ((n + 1 - my_rank) / n) * 100
        mean_percentile = np.mean(percentile_rank)
        
        print(f"{n:<15} | {win_rate:>13.2f}% | {mean_percentile:>18.2f}%")

if __name__ == '__main__':
    simulate_full_tournament()

Opponents (N)   | Outright Win %  | Mean Rank Percentile
-------------------------------------------------------
10              |         55.57% |              92.19%
50              |         25.27% |              92.15%
100             |         15.89% |              92.08%
500             |          4.85% |              92.00%
1000            |          2.66% |              92.10%
